# Day 35: Drift Detection and Optimization
**Drift types and mitigation, quality optimisation and LLMOps automation**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.

## 1. Setup

In [ ]:
# ===== Option A: Google Colab (active) =====
# Add your key once: click the key icon (Secrets) in the left sidebar, name it OPENAI_API_KEY, and allow notebook access.
!pip install -q openai scikit-learn numpy pandas scipy matplotlib

# ===== Option B: VS Code / local =====
# pip install openai scikit-learn numpy pandas scipy matplotlib
# Create a .env file or set the variable in your terminal:  OPENAI_API_KEY=sk-...
# No key? Everything still runs in OFFLINE demo mode with a tiny built-in stand-in model.
print("Setup done")

In [ ]:
%%writefile genai_helpers.py
"""Shared helpers for the GenAI days.
llm()   -> calls OpenAI when a key is available, otherwise a small OFFLINE stand-in so every cell still runs.
embed() -> OpenAI embeddings online, a hashing-based stand-in offline (same API, lower quality).
"""
import json, os, re
import numpy as np

def _get_key():
    try:
        from google.colab import userdata          # Colab Secrets
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY = _get_key()
ONLINE = bool(OPENAI_API_KEY)
MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")                 # change to any current chat model you have access to
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
if ONLINE:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

WORD = re.compile(r"[a-z0-9]+")

def _words(t):
    # crude normalisation so "expires" matches "expire"
    return {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in WORD.findall(t.lower()) if len(w) > 3}

def offline_llm(prompt, system=""):
    """A deliberately simple stand-in. It is NOT intelligent; it lets the code paths run without a key."""
    low = prompt.lower()
    if "context:" in low and "question:" in low:
        ctx = re.split(r"(?i)context:", prompt, 1)[1]
        ctx, q = re.split(r"(?i)question:", ctx, 1)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ctx.strip()) if s.strip()]
        best = max(sents, key=lambda s: len(_words(s) & _words(q)), default="")
        return best if best and _words(best) & _words(q) else "I don't know based on the provided context."
    if "classify" in low and "positive" in low:
        neg = sum(w in low for w in ["slow", "bad", "broken", "late", "angry", "worst", "refund"])
        pos = sum(w in low for w in ["great", "love", "fast", "thanks", "excellent", "happy"])
        return "negative" if neg > pos else "positive" if pos > neg else "neutral"
    if "json" in low:
        text = prompt.strip().splitlines()[-1]
        return json.dumps({"summary": " ".join(text.split()[:12]), "offline": True})
    first = re.split(r"(?<=[.!?])\s+", prompt.strip())[0]
    return f"[offline demo] {first[:160]}"

def llm(prompt, system="You are a helpful assistant.", temperature=0.2, model=None):
    if ONLINE:
        r = client.chat.completions.create(
            model=model or MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}])
        return r.choices[0].message.content
    return offline_llm(prompt, system)

def embed(texts):
    """Return an (n, d) array of unit-length vectors."""
    if isinstance(texts, str):
        texts = [texts]
    if ONLINE:
        r = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in r.data])
    else:
        from sklearn.feature_extraction.text import HashingVectorizer
        hv = HashingVectorizer(n_features=1024, ngram_range=(1, 2), alternate_sign=False, norm=None)
        v = hv.transform([t.lower() for t in texts]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(n == 0, 1, n)

print("Mode:", f"ONLINE ({MODEL})" if ONLINE else "OFFLINE demo (add OPENAI_API_KEY to use a real model)")

In [ ]:
import importlib, genai_helpers
importlib.reload(genai_helpers)
from genai_helpers import llm, embed, ONLINE

## 2. Reference and current periods with planted drift

In [ ]:
import json, re, sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import ks_2samp
rng = np.random.default_rng(35)
TEMPLATES = {
    "leave": ["How many leave days do I get", "When does carried forward leave expire", "Can I take sick leave without a certificate"],
    "it": ["How do I reset my password", "What does error VPN-417 mean", "When will my new laptop arrive"],
    "travel": ["What is the hotel limit in Japan", "How early should I book flights", "How do I submit travel expenses"],
    "visa": ["How do I get a visa support letter for Japan", "What documents do I need for a business visa", "How long does visa processing take"],
}
def make_week(week, n=600):
    visa = 0.0 if week < 4 else 0.25
    ja = 0.10 if week < 4 else 0.25
    topics = rng.choice(list(TEMPLATES), n, p=[0.4 - visa / 3, 0.3 - visa / 3, 0.3 - visa / 3, visa])
    rows = []
    for t in topics:
        q = rng.choice(TEMPLATES[t])
        if week == 4 and rng.random() < 0.5:
            q += " and what should I do if my manager is travelling that week"           # longer questions in week 4
        rows.append({"week": week, "topic": t, "lang": "ja" if rng.random() < ja else "en", "question": q,
                     "q_len": len(q.split()), "answer_tokens": rng.normal(170 if week < 4 else 240, 30)})
    return pd.DataFrame(rows)
data = pd.concat([make_week(w) for w in [1, 2, 3, 4]], ignore_index=True)
print(data.groupby("week").agg(q_len=("q_len", "mean"), answer_tokens=("answer_tokens", "mean")).round(1))

## 3. Detecting drift: PSI, KS test, categorical PSI

In [ ]:
def psi(ref, cur, bins=10):
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    r = np.histogram(ref, edges)[0] / len(ref) + 1e-6
    c = np.histogram(cur, edges)[0] / len(cur) + 1e-6
    return float(np.sum((c - r) * np.log(c / r)))

def psi_cat(ref, cur):
    cats = sorted(set(ref) | set(cur))
    r = np.array([(ref == k).mean() for k in cats]) + 1e-3          # smoothing: a brand new category still gives a large PSI
    c = np.array([(cur == k).mean() for k in cats]) + 1e-3
    return float(np.sum((c - r) * np.log(c / r)))

w1, w4 = data[data.week == 1], data[data.week == 4]
half = len(w1) // 2
print(f"PSI question length  week1 vs week4: {psi(w1.q_len, w4.q_len):.2f}")
print(f"PSI question length  week1 half A vs B (sanity check): {psi(w1.q_len[:half], w1.q_len[half:]):.2f}")
print(f"PSI topic            week1 vs week4: {psi_cat(w1.topic, w4.topic):.2f}")
print(f"PSI language         week1 vs week4: {psi_cat(w1.lang, w4.lang):.2f}")
ks = ks_2samp(w1.answer_tokens, w4.answer_tokens)
print(f"KS test answer tokens: statistic {ks.statistic:.2f}, p-value {ks.pvalue:.1e}  (small p-value = distributions differ)")

## 4. Embedding drift

In [ ]:
def centroid(texts):
    v = embed(list(texts)).mean(axis=0)
    return v / np.linalg.norm(v)

ref = centroid(w1.question.sample(200, random_state=1))
drift = {w: round(1 - float(ref @ centroid(data[data.week == w].question.sample(200, random_state=2))), 3) for w in [1, 2, 3, 4]}
print("Centroid drift vs week 1:", drift)
pd.Series(drift).plot(marker="o", color="#FF8A3D", title="Question embedding drift vs week 1")
plt.xlabel("week"); plt.ylabel("1 - cosine"); plt.show()

## 5. Concept drift: the golden set must change too

In [ ]:
golden_old = {"q": "What is the refund period?", "must_include": ["30 days"]}
golden_new = {"q": "What is the refund period?", "must_include": ["14 days"], "must_not_include": ["30 days"], "version": 2, "changed": "policy update 2026-12-01"}
def passes(g, answer):
    return all(m in answer for m in g["must_include"]) and not any(m in answer for m in g.get("must_not_include", []))
stale_answer = "Refunds are accepted within 30 days."
print("Old golden item, stale answer: ", "PASSES (wrong!)" if passes(golden_old, stale_answer) else "fails")
print("New golden item, stale answer: ", "passes" if passes(golden_new, stale_answer) else "FAILS (correct)")

## 6. Optimisation experiments: change one thing at a time

In [ ]:
VISA_DOC = ("## Visa support\nVisa support letters for Japan are issued by HR within 3 working days. "
            "Business visas need a passport, an invitation letter and the HR support letter. Processing usually takes 5 to 10 working days.")
def build_index(text):
    chunks = [s.strip() for s in re.split(r"(?<=[.!?])\s+", re.sub(r"#+ .*\n", "", text)) if s.strip()]
    return chunks, embed(chunks)

GOLDEN = [("How many sick days do I get?", "8 days of paid sick leave"), ("When does carried forward leave expire?", "31 March"),
          ("What does error VPN-417 mean?", "VPN-417"), ("Hotel limit per night in Japan?", "15,000 JPY"),
          ("Deadline for travel expenses?", "within 30 days"), ("How long must my password be?", "14 characters"),
          ("How early should I book flights?", "10 days before travel"),
          ("How do I get a visa support letter for Japan?", "Visa support letters"), ("What documents do I need for a business visa?", "invitation letter"),
          ("How long does visa processing take?", "5 to 10 working days")]

def hit_rate(chunks, vecs, k):
    hits = 0
    for q, frag in GOLDEN:
        top = np.argsort(vecs @ embed(q)[0])[::-1][:k]
        hits += any(frag in chunks[i] for i in top)
    return hits / len(GOLDEN)
print(len(GOLDEN), "golden questions, including 3 about visas")

In [ ]:
%%writefile knowledge.md
# Sakura Logistics Employee Handbook (sample)

## 1. Working hours
Standard working hours are 9:30 to 18:30, Monday to Friday. Teams in the Tokyo office follow JST and teams in the Bengaluru office follow IST.
Flexible start times between 8:00 and 11:00 are allowed with manager approval.

## 2. Leave policy
Full-time employees receive 18 days of paid leave per year. Leave accrues monthly at 1.5 days.
Unused leave can be carried forward up to a maximum of 10 days. Carried forward leave expires on 31 March of the following year.
Sick leave is separate: employees receive 8 days of paid sick leave per year. A medical certificate is required for sick leave longer than 2 consecutive days.

## 3. Remote work
Employees may work remotely up to 2 days per week. Remote work must be logged in the HR portal by 10:00 on the same day.
Remote work from another country requires approval from HR and the security team at least 14 days in advance.

## 4. Expenses
Travel expenses must be submitted within 30 days with receipts. Meals during business travel are reimbursed up to 2,000 INR or 4,000 JPY per day.
Expenses above 50,000 INR require approval from a director before booking.

## 5. Security
Laptops must use full disk encryption. Customer data must never be copied to personal devices or uploaded to unapproved AI tools.
Security incidents must be reported to security@sakura-logistics.example within 1 hour of discovery.

## IT FAQ
Error VPN-417 means your VPN certificate has expired. Renew it from the IT self-service portal and restart the VPN client.
Password resets are done through the self-service portal. Passwords must be at least 14 characters long.
Approved AI tools are listed on the IT intranet page. Do not paste customer data into tools that are not on that list.
New laptops are issued within 5 working days of a request approved by your manager.

## Travel FAQ
Book flights through the corporate travel desk at least 10 days before travel. Economy class is standard for flights under 6 hours.
Hotel stays are reimbursed up to 6,000 INR per night in India and 15,000 JPY per night in Japan.
Visa support letters are issued by HR within 3 working days.

In [ ]:
base = build_index(open("knowledge.md").read())
fixed = build_index(open("knowledge.md").read() + "\n\n" + VISA_DOC)
results = {"baseline (k=3)": hit_rate(*base, 3), "k=5": hit_rate(*base, 5),
           "add visa document (k=3)": hit_rate(*fixed, 3), "visa document + k=5": hit_rate(*fixed, 5)}
for name, r in results.items():
    print(f"{name:<26} hit rate {r:.2f}")
print("\nThe data fix (adding the missing document) beats tuning k.")

## 7. Automating it: a weekly LLMOps job

In [ ]:
%%writefile weekly_job.py
import json, sys
import numpy as np, pandas as pd

THRESHOLDS = {"psi_topic": 0.25, "psi_length": 0.25, "golden_hit_rate": 0.9}

def psi_cat(ref, cur):
    cats = sorted(set(ref) | set(cur))
    r = np.array([(ref == k).mean() for k in cats]) + 1e-3
    c = np.array([(cur == k).mean() for k in cats]) + 1e-3
    return float(np.sum((c - r) * np.log(c / r)))

def psi_num(ref, cur, bins=10):
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    r = np.histogram(ref, edges)[0] / len(ref) + 1e-3
    c = np.histogram(cur, edges)[0] / len(cur) + 1e-3
    return float(np.sum((c - r) * np.log(c / r)))

def main(ref_path, cur_path, golden_hit_rate, week):
    ref, cur = pd.read_csv(ref_path), pd.read_csv(cur_path)
    m = {"psi_topic": round(psi_cat(ref.topic, cur.topic), 2),
         "psi_length": round(psi_num(ref.q_len, cur.q_len), 2),
         "golden_hit_rate": golden_hit_rate}
    breaches = [k for k, v in m.items() if (v > THRESHOLDS[k] if k.startswith("psi") else v < THRESHOLDS[k])]
    lines = [f"# LLMOps weekly report {week}", "", "| Metric | Value | Threshold | Status |", "|---|---|---|---|"]
    lines += [f"| {k} | {v} | {THRESHOLDS[k]} | {'REVIEW' if k in breaches else 'ok'} |" for k, v in m.items()]
    lines += ["", "Action: " + ("review the breached metrics this week." if breaches else "no action needed.")]
    open(f"report_{week}.md", "w").write("\n".join(lines))
    print("\n".join(lines))
    return 1 if breaches else 0

if __name__ == "__main__":
    sys.exit(main(sys.argv[1], sys.argv[2], float(sys.argv[3]), sys.argv[4]))

In [ ]:
w1.to_csv("week1.csv", index=False); w4.to_csv("week4.csv", index=False)
import subprocess
r = subprocess.run([sys.executable, "weekly_job.py", "week1.csv", "week4.csv", str(results["baseline (k=3)"]), "2026-W52"],
                   capture_output=True, text=True)
print(r.stdout)
print("exit code:", r.returncode, "(1 means a person should review; a scheduler would open a ticket or alert)")

# ===== Option B: schedule it =====
# Azure DevOps: a pipeline with  schedules: - cron: "0 3 * * 1"  that runs this script
# Linux cron:   0 3 * * 1  python weekly_job.py week_ref.csv week_cur.csv 0.95 $(date +%G-W%V)

### TODO 1: Add the KS test on answer tokens to `weekly_job.py` with a p-value threshold of 0.01.

In [ ]:
# TODO: write your code here

---
**Yoi shuumatsu wo** (良い週末を): *Have a great weekend*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)